In [ ]:
from clickhouse_driver import Client
import os
import pandas as pd

# Switch to use LocalXpose tunnel for off-campus access
USE_LOCALXPOSE = True  # Set to True for off-campus access

if USE_LOCALXPOSE:
    # LocalXpose tunnel (off-campus)
    CH_HOST = os.environ.get('CH_HOST', 'ap.loclx.io')
    CH_PORT = int(os.environ.get('CH_PORT', '41304'))
    CH_SECURE = False
else:
    # Direct connection (on-campus)
    CH_HOST = os.environ.get('CH_HOST', 'chenlin04.fbe.hku.hk')
    CH_PORT = int(os.environ.get('CH_PORT', '9000'))
    CH_SECURE = False

CH_USER = os.environ.get('CH_USER', 'yaode_xie')


token_file = os.path.expanduser('~/clickhouse_token.txt')
if os.path.exists(token_file):
    with open(token_file, 'r') as f:
        password = f.read().strip()
    print(f'Read password from {token_file}')
else:
    password = "3036555659"
    print(f'Using default password')

client = Client(
    host=CH_HOST,
    port=CH_PORT,
    user=CH_USER,
    password=password,
    secure=CH_SECURE,
)

def q(sql):
    rows, cols = client.execute(sql, with_column_types=True)
    return pd.DataFrame(rows, columns=[column[0] for column in cols])


try:
    client.execute('SELECT 1')
    print(f'✅ Connected to ClickHouse as {CH_USER} on {CH_HOST}:{CH_PORT}')
except Exception as exc:
    raise RuntimeError(
        f'ClickHouse connection failed. '
        f'Check credentials or ensure you are on campus network. '
        f'Error: {exc}'
    ) from exc

Using default password
✅ Connected to ClickHouse as yaode_xie on ap.loclx.io:41304


In [91]:
q('SHOW DATABASES')

,name
0,binance
1,common_goods
2,comp_202601
3,crsp_202501
4,ff
5,hyperliquid
6,ibes_202601
7,mfin7035_shared_write
8,tr_sdc_ma_202601
9,tr_sdc_ni_202601


In [92]:
df = q('''Select * from ibes_202601.det_epsus limit 10
''')

In [93]:
df.columns

Index(['ticker', 'cusip', 'oftic', 'cname', 'actdats', 'estimator', 'analys',
       'currfl', 'pdf', 'fpi', 'measure', 'value', 'curr', 'usfirm', 'fpedats',
       'acttims', 'revdats', 'revtims', 'anndats', 'anntims', 'actual',
       'actdats_act', 'acttims_act', 'anndats_act', 'anntims_act', 'curr_act',
       'report_curr'],
      dtype='object')

In [82]:
q('''WITH us_analysts AS (
    SELECT DISTINCT analys, ticker
    FROM ibes_202601.det_epsus
    WHERE analys IS NOT NULL
      AND report_curr = 'USD'
),
jp_analysts AS (
    SELECT DISTINCT analys, ticker
    FROM ibes_202601.det_epsint
    WHERE analys IS NOT NULL
      AND report_curr = 'JPY'
)
SELECT 
    a.analys,
    COUNT(DISTINCT a.ticker) as us_stocks,
    COUNT(DISTINCT b.ticker) as jp_stocks,
    COUNT(DISTINCT a.ticker) + COUNT(DISTINCT b.ticker) as total_coverage
FROM us_analysts a
JOIN jp_analysts b ON a.analys = b.analys
GROUP BY a.analys
HAVING us_stocks > 0 AND jp_stocks > 0
ORDER BY total_coverage DESC
''')

,analys,us_stocks,jp_stocks,total_coverage
0,0.0,1878,6120,7998
1,200938.0,1044,2,1046
2,22084.0,167,467,634
3,135374.0,207,1,208
4,51283.0,1,188,189
...,...,...,...,...
832,108317.0,1,1,2
833,105098.0,1,1,2
834,133982.0,1,1,2
835,122771.0,1,1,2


In [ ]:
q('''WITH 
cross_border AS (
    SELECT 
        a.analys,
        COUNT(DISTINCT a.ticker) as us_stocks,
        COUNT(DISTINCT b.ticker) as jp_stocks
    FROM (
        SELECT DISTINCT analys, ticker
        FROM ibes_202601.det_epsus
        WHERE analys IS NOT NULL AND report_curr = 'USD'
    ) a
    JOIN (
        SELECT DISTINCT analys, ticker
        FROM ibes_202601.det_epsint
        WHERE analys IS NOT NULL AND report_curr = 'JPY'
    ) b ON a.analys = b.analys
    GROUP BY a.analys
)
SELECT 
    COUNT(*) as total_cross_border_analysts,
    AVG(us_stocks) as avg_us_coverage,
    AVG(jp_stocks) as avg_jp_coverage,
    MAX(us_stocks) as max_us_coverage,
    MAX(jp_stocks) as max_jp_coverage,
    COUNT(CASE WHEN us_stocks >= 5 AND jp_stocks >= 5 THEN 1 END) as analysts_with_5plus_both
FROM cross_border''')

,total_cross_border_analysts,avg_us_coverage,avg_jp_coverage,max_us_coverage,max_jp_coverage,analysts_with_5plus_both
0,837,15.702509,20.389486,1878,6120,196
